# Cohezion Name That LLM: SOTA v6 Ensembled Temperature-Scaled Sinkhorn Balancing

**Competition**: [Name That LLM](https://www.kaggle.com/competitions/name-that-llm) ($50,000 USD Featured Track)
**Evaluation Metric**: `Balanced Accuracy Score` across 18 Generative LLM classes

### SOTA v6 Architecture:
1. **Logit Temperature Scaling ($T=0.85$)**: Insulates multi-class cross-entropy from extreme logit penalization on unobserved topics.
2. **Leak-Free Group Invariant**: `StratifiedGroupKFold(n_splits=5, groups=prompt_id)` prevents the 17 sibling model answers per prompt from leaking between train and calibration folds.
3. **Dual Heterogeneous Ensemble**: Calibrated LinearSVC + HistGradientBoostingClassifier on 128-dim TruncatedSVD components.
4. **Sinkhorn Uniform Marginal Balancing**: Solves optimal transport dual variables to enforce exact $1/18$ model distribution on test.


In [ ]:
# Cell 1: Environment & Deterministic Setup
import os
import sys
import math
import pathlib
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✓ Environment configured with seed=42.')


In [ ]:
# Cell 2: Robust Kaggle Path Discovery
CANDIDATES = [
    pathlib.Path('/kaggle/input/name-that-llm'),
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('data/name_that_llm'),
    pathlib.Path('../data/name_that_llm'),
]

DATA_DIR = next((p for p in CANDIDATES if (p / 'train.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate name-that-llm dataset!')

print(f'Data root resolved to: {DATA_DIR}')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

classes = sorted(train_df['model'].unique())
n_classes = len(classes)
print(f'Train samples: {len(train_df)} | Classes: {n_classes} | Unique prompts: {train_df["prompt_id"].nunique()}')
print(f'Test samples: {len(test_df)}')


In [ ]:
# Cell 3: First-Principles Compression Entropy, Structural & Stylometric Feature Extraction
import bz2
import lzma
import re
import string
import zlib
import numpy as np
import pandas as pd

FUNCTION_WORDS = [
    'the', 'of', 'and', 'to', 'a', 'in', 'that', 'is', 'was', 'he',
    'for', 'it', 'with', 'as', 'his', 'on', 'be', 'at', 'by', 'i',
    'this', 'had', 'not', 'are', 'but', 'from', 'or', 'have', 'an', 'they',
    'which', 'one', 'you', 'were', 'her', 'all', 'she', 'there', 'would',
    'their', 'we', 'him', 'been', 'has', 'when', 'who', 'will', 'more', 'no',
    'if', 'out', 'so', 'said', 'what', 'up', 'its', 'about', 'into', 'than',
    'them', 'can', 'only', 'other', 'new', 'some', 'could', 'time', 'these',
    'two', 'may', 'then', 'do', 'first', 'any', 'my', 'now', 'such', 'like',
    'our', 'over', 'man', 'me', 'even', 'most', 'made', 'after', 'also', 'did',
    'many', 'before', 'must', 'through', 'back', 'years', 'where', 'much', 'your',
    'way', 'well', 'down', 'should', 'because', 'each', 'just', 'those', 'people',
    'crucially', 'moreover', 'furthermore', 'thus', 'however', 'nonetheless',
    'indeed', 'essentially', 'specifically', 'ultimately', 'additionally',
    'certainly', 'sure', 'overall', 'firstly', 'secondly', 'finally',
]

def extract_compression_and_stylometric_features(texts):
    rows = []
    for text in texts:
        t_str = str(text) if text is not None else ''
        raw_bytes = t_str.encode('utf-8', errors='replace')
        n_bytes = max(1, len(raw_bytes))
        chars = len(t_str)
        words = t_str.split()
        num_words = max(1, len(words))
        sentences = [s.strip() for s in re.split(r'[.!?]+', t_str) if s.strip()]
        num_sentences = max(1, len(sentences))

        # Compression ratios
        cr_zlib = len(zlib.compress(raw_bytes, level=6)) / n_bytes
        cr_bz2 = len(bz2.compress(raw_bytes, compresslevel=6)) / n_bytes
        cr_lzma = len(lzma.compress(raw_bytes)) / n_bytes

        # Byte entropy
        byte_counts = np.bincount(np.frombuffer(raw_bytes, dtype=np.uint8), minlength=256)
        probs = byte_counts[byte_counts > 0] / n_bytes
        byte_entropy = -float(np.sum(probs * np.log2(probs)))

        # Structural & formatting
        lines = t_str.splitlines()
        n_lines = max(1, len(lines))
        star_bullets = sum(1 for line in lines if line.strip().startswith('* ')) / n_lines
        dash_bullets = sum(1 for line in lines if line.strip().startswith('- ')) / n_lines
        num_bullets = sum(1 for line in lines if re.match(r'^\d+\.\s', line.strip())) / n_lines

        h1 = t_str.count('\n# ') + (1 if t_str.startswith('# ') else 0)
        h2 = t_str.count('\n## ') + (1 if t_str.startswith('## ') else 0)
        h3 = t_str.count('\n### ') + (1 if t_str.startswith('### ') else 0)
        header_ratio = (h1 + h2 + h3) / n_lines

        code_blocks = t_str.count('```') / 2.0
        inline_code = t_str.count('`') - (code_blocks * 6)

        avg_word_len = chars / num_words
        avg_sent_len = num_words / num_sentences
        unique_words = len(set(w.lower() for w in words))
        ttr = unique_words / num_words

        first_50 = t_str[:100].lower()
        opener_certainly = 1.0 if 'certainly' in first_50 else 0.0
        opener_sure = 1.0 if 'sure' in first_50 else 0.0
        opener_here = 1.0 if ('here is' in first_50 or "here's" in first_50) else 0.0
        think_tag = 1.0 if ('<think>' in t_str or '</think>' in t_str) else 0.0

        t_lower = t_str.lower()
        words_lower = [w.strip(string.punctuation) for w in t_lower.split()]
        word_set_counts = pd.Series(words_lower).value_counts().to_dict()
        fw_freqs = [word_set_counts.get(fw, 0) * 100.0 / num_words for fw in FUNCTION_WORDS]

        row = [
            cr_zlib, cr_bz2, cr_lzma, byte_entropy, float(chars), float(num_words),
            avg_word_len, avg_sent_len, ttr, star_bullets, dash_bullets, num_bullets,
            header_ratio, code_blocks, inline_code, opener_certainly, opener_sure,
            opener_here, think_tag,
        ] + fw_freqs
        rows.append(row)
    return np.nan_to_num(np.array(rows, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
# Cell 4: Model Training, Group-KFold Calibration, Dual Ensemble, Temperature Scaling & Sinkhorn Balancing
import pathlib
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier

print('=== TRAINING MODEL 1: Calibrated LinearSVC with Leak-Free StratifiedGroupKFold(groups=prompt_id) ===')
sgkf = StratifiedGroupKFold(n_splits=5)
prompt_groups = train_df['prompt_id'].values if 'prompt_id' in train_df.columns else None
if prompt_groups is not None:
    cv_splits = list(sgkf.split(X_train_sparse, y_train, groups=prompt_groups))
    print(f'Constructed {len(cv_splits)} leak-free prompt-grouped folds across {len(np.unique(prompt_groups))} unique prompts.')
else:
    cv_splits = 5
    print('Warning: prompt_id not found in train_df; falling back to standard 5-fold CV.')

base_svc = LinearSVC(C=0.45, max_iter=2500, random_state=42)
model_svc = CalibratedClassifierCV(estimator=base_svc, method='sigmoid', cv=cv_splits)
model_svc.fit(X_train_sparse, y_train)
svc_test_probs = model_svc.predict_proba(X_test_sparse)

print('=== TRAINING MODEL 2: HistGradientBoosting on 128-dim TruncatedSVD Latent Space ===')
svd = TruncatedSVD(n_components=128, random_state=42)
X_train_svd = np.hstack([svd.fit_transform(X_train_sparse), X_train_dense])
X_test_svd = np.hstack([svd.transform(X_test_sparse), X_test_dense])
model_hgb = HistGradientBoostingClassifier(max_iter=150, max_depth=6, random_state=42)
model_hgb.fit(X_train_svd, y_train)
hgb_test_probs = model_hgb.predict_proba(X_test_svd)

# Weighted consensus blending
test_probs = 0.60 * svc_test_probs + 0.40 * hgb_test_probs

# Grandmaster Invariant: Logit Temperature Scaling (T=0.85)
T = 0.85
logits = np.log(np.clip(test_probs, 1e-6, 1.0 - 1e-6))
scaled_probs = np.exp(logits / T)
scaled_probs /= scaled_probs.sum(axis=1, keepdims=True)

# Grandmaster Invariant: Sinkhorn-Knopp Uniform Marginal Balancing
n_samples, n_classes = scaled_probs.shape
target_priors = np.ones(n_classes) / n_classes  # Uniform 1/18 target
balanced_probs = scaled_probs.copy()
for _ in range(25):
    balanced_probs /= (balanced_probs.sum(axis=1, keepdims=True) + 1e-12)
    col_sums = balanced_probs.sum(axis=0) + 1e-12
    balanced_probs *= (target_priors / col_sums)
balanced_probs /= (balanced_probs.sum(axis=1, keepdims=True) + 1e-12)
balanced_probs = np.clip(balanced_probs, 1e-6, 1.0 - 1e-6)

# Build submission dataframe
class_labels = model_svc.classes_
sub_df = pd.DataFrame(balanced_probs, columns=class_labels)
sub_df.insert(0, 'id', test_df['id'].values)
sub_df.to_parquet('submission.parquet', index=False)
print(f'Successfully wrote {len(sub_df)} predictions to submission.parquet!')
